# Lab 2 — Synthetic memory bootstrap

Goal: provision useful operating context before an agent has real history. Synthetic memory is not ground truth about a user; it is a controlled way to seed workflows, entities, policies, skills, and tool descriptions for a demo or new domain.


## 0. Generate the bundle offline

The helper is deterministic by default and samples curated, distinct content without replacement. You can run this cell without Oracle or an LLM provider.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path
import json

from agentic_memory_workshop.synthesize_memory import generate_bundle, write_bundle

bundle = generate_bundle(
    user_id='workshop_user',
    agent_id='memory_agent',
    domain='restaurant concierge',
    thread_count=20,
    workflow_count=15,
    entity_count=30,
    knowledge_count=50,
    skill_count=6,
    tool_count=12,
    seed=7,
)
out = Path('../generated/synthetic_memory.json')
write_bundle(bundle, out, write_skill_files=True)
print(f'Wrote {out}')
print(f'Domain: {bundle.domain}, user: {bundle.user_id}, agent: {bundle.agent_id}')
print(f'Threads: {len(bundle.threads)}, workflows: {len(bundle.workflows)}, entities: {len(bundle.entities)}')
print(f'Knowledge: {len(bundle.knowledge)}, skills: {len(bundle.skills)}, tools: {len(bundle.tools)}')

## 1. Inspect what was generated

The bundle covers each memory category we want to seed: conversational/episodic threads, workflows, entities, semantic knowledge, skills, and tools.

In [ ]:
data = json.loads(out.read_text())
for key in ['threads', 'workflows', 'entities', 'knowledge', 'skills', 'tools']:
    print(f'{key:10s}', len(data[key]))

thread = data['threads'][0]
print(f"\nThread {thread['thread_key']} ({thread['topic']}):")
for m in thread['messages']:
    print(f"  {m['role']:9s} {m['content']}")

print('\nWorkflow:', data['workflows'][0]['content'])
print('\nEntity:', data['entities'][0]['content'])
print('\nKnowledge:', data['knowledge'][0]['content'])

skill = data['skills'][0]
print(f"\nSkill {skill['name']}:")
print('\n'.join(skill['body'].splitlines()[:6]))

tool = data['tools'][0]
print(f"\nTool {tool['name']}: {tool['description']}")
print('  parameters:', ', '.join(tool['parameters']['properties']), '| required:', tool['parameters']['required'])

## 2. Load synthetic memory into OAMP

The load uses the same API surface as Lab 1, written out in the next cells: `add_user`, `add_agent`, `create_thread`, `thread.add_messages`, and `add_memory`. Every record has a stable id and is deleted before it is added, so rerunning this section replaces records instead of duplicating them. With `OAMP_EMBEDDER=oracle-db`, embeddings run inside Oracle.

In [ ]:
from agentic_memory_workshop.connection import connect, load_env
from agentic_memory_workshop.onnx import oamp_embedder
from oracleagentmemory.apis.thread import Message
from oracleagentmemory.core import OracleAgentMemory

load_env('../.env')
conn = connect()
client = OracleAgentMemory(
    connection=conn,
    embedder=oamp_embedder(conn),
    extract_memories=False,
    schema_policy='create_if_necessary',
)
print('Connected to Oracle and created the OAMP client')

for add, entity_id, info in [
    (client.add_user, bundle.user_id, f'Synthetic workshop user for {bundle.domain}.'),
    (client.add_agent, bundle.agent_id, f'Synthetic workshop agent for {bundle.domain}.'),
]:
    try:
        add(entity_id, info)
        print(f'Created {entity_id}')
    except ValueError as exc:
        if 'already exists' not in str(exc):
            raise
        print(f'{entity_id} already exists, reusing it')

In [ ]:
for item in bundle.threads:
    client.delete_thread(item.thread_key)
    thread = client.create_thread(
        thread_id=item.thread_key, user_id=bundle.user_id, agent_id=bundle.agent_id
    )
    thread.add_messages([Message(role=m.role, content=m.content) for m in item.messages])

message_total = sum(len(item.messages) for item in bundle.threads)
print(f'Loaded {len(bundle.threads)} threads with {message_total} messages')
print(f'Thread ids: {bundle.threads[0].thread_key} .. {bundle.threads[-1].thread_key}')

In [ ]:
records = [*bundle.workflows, *bundle.entities, *bundle.knowledge]
counts = {}
for record in records:
    memory_id = f'synthetic-{record.subject_key}'
    client.delete_memory(memory_id)
    client.add_memory(
        f'[{record.record_type}] {record.content}',
        user_id=bundle.user_id,
        agent_id=bundle.agent_id,
        memory_id=memory_id,
    )
    counts[record.record_type] = counts.get(record.record_type, 0) + 1

print(f'Loaded {len(records)} memories: {counts}')
print(f'Example memory id: synthetic-{records[0].subject_key}')
print('Ids are stable, so running this section again replaces these records instead of duplicating them.')

## 3. Prove first-turn retrieval

Now create a fresh client and ask questions that should retrieve synthetic context even though this thread has no real conversation history. Results are listed by type, with duplicate contents removed.

In [ ]:
search_client = OracleAgentMemory(
    connection=conn,
    embedder=oamp_embedder(conn),
    extract_memories=False,
    schema_policy='create_if_necessary',
)

def show(query, top=5):
    print(f'Query: {query}')
    seen = set()
    for result in search_client.search(query, user_id='workshop_user', max_results=top * 3):
        if result.content in seen:
            continue
        seen.add(result.content)
        print(f'  [{result.record.record_type}] distance={result.distance:.3f}')
        print(f'    {result.content[:220]}')
        if len(seen) == top:
            break
    print(f'{len(seen)} distinct results')

show('How should the agent handle a refund request for a restaurant booking?')

In [ ]:
show('Which table suits a guest with a peanut allergy who likes a quiet seat?')

## 4. Optional: load synthetic tools and skills

Run this after the in-database ONNX embedding model is available. It seeds the same registries used in the mini-lab.

In [ ]:
from agentic_memory_workshop.skillbox import retrieve_skills, upsert_skill
from agentic_memory_workshop.toolbox import retrieve_tools, upsert_tool

try:
    for tool in bundle.tools:
        upsert_tool(conn, tool.name, tool.description, tool.parameters, source='synthetic')
    print(f'Upserted {len(bundle.tools)} tools:', ', '.join(t.name for t in bundle.tools))

    for skill in bundle.skills:
        upsert_skill(conn, skill.name, skill.body, description=skill.description, metadata='{"source":"synthetic"}')
    print(f'Upserted {len(bundle.skills)} skills:', ', '.join(s.name for s in bundle.skills))

    query = 'A guest wants to cancel a booking and get their deposit back'
    print(f'\nTop 3 tools for: {query}')
    for row in retrieve_tools(conn, query, k=3):
        print(f"  {row['name']} distance={row['distance']:.3f}")
    print('Top 3 skills for the same query:')
    for row in retrieve_skills(conn, query, k=3):
        print(f"  {row['name']} distance={row['distance']:.3f}")
except Exception as exc:
    print('Toolbox/skillbox load skipped or failed. Most common cause: ONNX model not loaded.')
    print(type(exc).__name__, exc)